# 07 - Final Evaluation & Error Analysis

Project: **RoadEye - Traffic Sign Recognition Using HOG Features and SVM** (Machine Learning Innovative Assignment).

> Run cells top to bottom. All numbers are computed when the cell runs; nothing is typed in by hand.

In [ ]:
import sys, pathlib, json, warnings
warnings.filterwarnings("ignore")
ROOT = pathlib.Path.cwd().resolve()
ROOT = ROOT.parent if ROOT.name == "notebooks" else ROOT
sys.path.insert(0, str(ROOT))
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from IPython.display import display, Image, Markdown
from src import config, pipeline as P
np.random.seed(config.SEED)                      # reproducibility
pd.set_option("display.width", 160); pd.set_option("display.max_columns", 30)
print("RUN MODE:", config.RUN_MODE, "| seed:", config.SEED)

In [ ]:
def show(name, width=900):
    """Display a saved figure from results/figures/."""
    display(Image(filename=str(config.FIGURES_DIR / name), width=width))

def metrics(name):
    return json.loads((config.METRICS_DIR / name).read_text())

## One-shot test evaluation
The frozen models are scored on the untouched test set **once**. A lock file stores the model hash: re-running this notebook replays the stored predictions and refuses to evaluate a *changed* model (that would be tuning on the test set).

In [ ]:
data = P.stage_prepare(save_figures=False)
res = P.stage_final_eval(data)
display(pd.Series({k: res[k] for k in ['model','test_source','n_test','accuracy','precision_macro','recall_macro','f1_macro','f1_weighted','balanced_accuracy','validation_accuracy','train_accuracy','generalisation_gap_val_minus_test_accuracy']}))

In [ ]:
display(pd.read_csv(config.METRICS_DIR / 'final_test_all_models.csv').round(4)[['model','accuracy','precision_macro','recall_macro','f1_macro','balanced_accuracy']])
show('model_performance_comparison_test.png', 800)

## Confusion matrix and per-class performance

In [ ]:
show('final_confusion_matrix.png', 750); pc = pd.read_csv(config.METRICS_DIR / 'final_test_per_class.csv'); display(pc.sort_values('f1').head(10).round(3)); show('final_per_class_f1.png', 600)

## Error analysis

In [ ]:
tc = pd.read_csv(config.METRICS_DIR / 'final_test_top_confusions.csv'); display(tc)
show('misclassified_examples.png', 900)

In [ ]:
z = np.load(config.METRICS_DIR / 'test_predictions.npz'); yt, yp = z['y_true'], z['final_pred']
wrong = yt != yp
speed = set(range(0, 9))  # GTSRB speed-limit family (classes 0-8)
both_speed = np.mean([(t in speed and p in speed) for t, p in zip(yt[wrong], yp[wrong])]) if wrong.any() else float('nan')
print(f'errors: {wrong.sum()} of {len(yt)}; share of errors that are speed-limit <-> speed-limit confusions: {both_speed:.1%}')
imgs = data.images[data.idx('test')]; br = imgs.reshape(len(imgs), -1).mean(1)
print('mean brightness of misclassified vs correct images:', round(br[wrong].mean(), 1), round(br[~wrong].mean(), 1))

Systematic patterns to discuss in the report: confusions inside the speed-limit family (digit shapes differ by a few pixels at 32x32), dark/low-contrast images, and very small signs. Never retune after looking at these results.